# Compare a grid of 25 physical designs

This is a direct solver study, without AI. Five damping ratios and five adhesive thicknesses make 25 cases. The full example took about two minutes on the recorded CPU, plus movie rendering. The runner screens every case and reruns three finalists at finer timesteps.

Two one-second MATLAB cases, with original function generation and ode45, imply about 32 minutes for all 25 cases before plotting/videos. This is a two-case estimate, not a measurement of the whole MATLAB grid; details are in docs/performance.md.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the repository or tutorials directory."
os.chdir(ROOT)
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/walking-robot-mpl")
print("Repository:", ROOT)
print("Python:", sys.executable)


In [ ]:
import yaml
from IPython.display import Image, display
CONFIG = Path("examples/sweeps/supported_25_cases.yaml")
configuration = yaml.safe_load(CONFIG.read_text())
print(yaml.safe_dump(configuration, sort_keys=False))
count = 1
for values in configuration["parameters"].values():
    count *= len(values)
print("Number of designs:", count)
OUTPUT = Path("outputs/tutorial_sweep")


Each list supplies one axis; all combinations are simulated. Thickness is in micrometers and damping ratio is dimensionless. `drive_frequency_hz: null` follows MATLAB's original design-dependent rule. Use a separate YAML with `363.0` for fixed-frequency comparisons.

This example uses zero initial clearance. The separate `matlab_25_cases.yaml` reproduces the original 501 µm drop, which exceeded our provisional penetration limit in all recorded cases. Changing the start changes the experiment.


In [ ]:
command = [sys.executable, "examples/run_sweep.py", "--config", str(CONFIG), "--output", str(OUTPUT), "--resume", "--render-best", "--export-mat"]
subprocess.run(command, check=True)
report = json.loads((OUTPUT / "report.json").read_text())
print("Passing screening cases:", report["screening_passed"], "/", report["total_cases"])
print("Best verified finalist:", report["best_id"])
print("Verified mean speed (m/s):", report["best_speed_m_s"])
display(Image(filename=str(OUTPUT / "sweep.png")))


In [ ]:
import csv
rows = list(csv.DictReader((OUTPUT / "results.csv").open()))
for row in rows:
    print(row)


Open `results.csv` in a spreadsheet. `physical_passed` refers to screening; `refinement_status` distinguishes checked finalists from cases that were not refined. A fast case with violations is excluded from selection. Set `verification.top_cases: 25` in a new configuration to refine every passing case.

If no finalist qualifies, `best_design.yaml` is absent. The report preserves the failed cases rather than inventing a winner. Otherwise, the selected design and `best_case.mat` can be used for an independent MATLAB check. The exported trajectory is reduced for storage; the integration timestep is recorded separately.


In [ ]:
RUN_MATLAB = False  # Optional; requires MATLAB with Symbolic Math Toolbox.
MATLAB_EXECUTABLE = "matlab"  # Or the full path to your MATLAB executable.
if RUN_MATLAB and report["best_id"] is not None:
    subprocess.run([sys.executable, "tools/check_sweep_matlab.py", "--design", str(OUTPUT / "best_design.yaml"), "--matlab", MATLAB_EXECUTABLE, "--timestep", "0.00000125", "--output", str(OUTPUT / "matlab_check.json")], check=True)
else:
    print("MATLAB check skipped. See docs/sweeps.md for its scope and commands.")
